In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, gc, math, pathlib
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

MODEL_DIR = "models/Qwen3-0.6B"
OUTPUT_DIR = "models/Qwen3-0.6B-W4A16"

if not os.path.isdir(MODEL_DIR):
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="Qwen/Qwen3-0.6B",
        local_dir="models/qwen3-0.6b"
    )

print(f"Base model:      {MODEL_DIR}")
print(f"Quantized model: {OUTPUT_DIR}")

Fetching 10 files: 100%|██████████| 10/10 [00:00<00:00, 9444.50it/s]

Base model:      models/Qwen3-0.6B
Quantized model: models/Qwen3-0.6B-W4A16


In [7]:
from llmcompressor.modifiers.quantization import GPTQModifier

recipe = GPTQModifier(
    scheme="W4A16",
    targets="Linear",
    ignore=["lm_head"],
)

print(f"Recipe: {recipe}")

Recipe: config_groups=None targets=['Linear'] ignore=['lm_head'] scheme='W4A16' kv_cache_scheme=None weight_observer=None input_observer=None output_observer=None observer=None bypass_divisibility_checks=False index=None group=None start=None end=None update=None initialized_=False finalized_=False started_=False ended_=False sequential_targets=None block_size=128 dampening_frac=0.01 actorder=static offload_hessians=False


In [ ]:
from llmcompressor import oneshot

if not os.path.isdir(OUTPUT_DIR):
    oneshot(
        model="Qwen/Qwen3-0.6B",
        dataset="wikitext",
        dataset_config_name="wikitext-2-raw-v1",
        recipe=recipe,
        output_dir=OUTPUT_DIR,
        max_seq_length=4096,
        num_calibration_samples=256,
    )
    print(f"Quantization complete. Model saved to: {OUTPUT_DIR}")